# Analiza streszczeń literatury młodzieżowej 

## Import bibliotek

In [10]:
import nltk 
import json
import string
import morfeusz2
import pandas as pd
import operator as op
import itertools as it
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader

In [2]:
nltk.download('punkt_tab')
nltk.pathsec.ENFORCE = False


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\smile\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Utworzenie korpusu dokumentów

In [3]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r" .* \.txt")
files_names = corpus.fileids()

## Wstępne przygotowanie dokumentów

In [4]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

{}


In [5]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print(json.dumps(lengths, indent=4, ensure_ascii=False))

{}


In [6]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:] #skip the first four lines of the file
stopwords_file.close()
stopwords

['vol',
 'o.o.',
 'mgr',
 'godz',
 'zł',
 'www',
 'pl',
 'ul',
 'tel',
 'hab',
 'prof',
 'inż',
 'dr',
 'i',
 'u',
 'aby',
 'albo',
 'ale',
 'ani',
 'aż',
 'bardzo',
 'bez',
 'bo',
 'bowiem',
 'by',
 'byli',
 'bym',
 'był',
 'była',
 'było',
 'były',
 'być',
 'będzie',
 'będą',
 'chce',
 'choć',
 'co',
 'coraz',
 'coś',
 'czy',
 'czyli',
 'często',
 'dla',
 'do',
 'gdy',
 'gdyby',
 'gdyż',
 'gdzie',
 'go',
 'ich',
 'im',
 'inne',
 'iż',
 'ja',
 'jak',
 'jakie',
 'jako',
 'je',
 'jednak',
 'jednym',
 'jedynie',
 'jego',
 'jej',
 'jest',
 'jeszcze',
 'jeśli',
 'jeżeli',
 'już',
 'ją',
 'kiedy',
 'kilku',
 'kto',
 'która',
 'które',
 'którego',
 'której',
 'który',
 'których',
 'którym',
 'którzy',
 'lat',
 'lecz',
 'lub',
 'ma',
 'mają',
 'mamy',
 'mi',
 'miał',
 'mimo',
 'mnie',
 'mogą',
 'może',
 'można',
 'mu',
 'musi',
 'na',
 'nad',
 'nam',
 'nas',
 'nawet',
 'nic',
 'nich',
 'nie',
 'niej',
 'nim',
 'niż',
 'no',
 'nowe',
 'np',
 'nr',
 'o',
 'od',
 'ok',
 'on',
 'one',
 'oraz',
 '

In [8]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group): 
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perpl , lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))
        

In [9]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join(char for char in documents_dict[file_name] if char not in string.punctuation)
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language = 'polish') if word not in stopwords])
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

{}


In [ ]:
for file_name in documents_dict:
    lengths [file_name]["post"] = len(word_tokenize(documents_dict[file_name])),
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [13]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percentage'] =((lengths['change'] / lengths['pre']) * 100).round(2)
lengths

KeyError: 'post'

In [14]:
documents_df = pd.DataFrame. from_dict(documents_dict, orient='index', columns=['content'])
documents_df

,content


In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df['content'])
matrix_tfidf = TfidfVectorizer().fit_transform(documents_df['content'])

In [ ]:
sparsity_tf =(1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf. shape[1]))) * 100
sparsity_tfidf =(1 - (matrix_tfidf.nnz / (matrix_tfidf.shape[0] * matrix_tfidf.shape[1]))) * 100